In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import json
import os
from functools import partial
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import nomad.io.base as loader
import nomad.stop_detection.utils as utils
import nomad.stop_detection.lachesis as LACHESIS
import nomad.stop_detection.sequential as SEQUENTIAL
from nomad.stop_detection.density_based import seqscan_labels
from nomad.stop_detection.dbscan import ta_dbscan_labels
from nomad.stop_detection.density_based import seqscan_labels_per_user
from nomad.stop_detection.lachesis import lachesis_labels_per_user

import nomad.visit_attribution.visit_attribution as visits

from nomad.contact_estimation import compute_stop_detection_metrics
import warnings
warnings.filterwarnings('ignore', message='Input is timezone-aware; assuming UTC')

In [2]:

def user_maxima_summary(data, algo_prefix, beta_ping, ha, metric="recall", param='delta_roam'):
    data = data.loc[data.algorithm.str.startswith(algo_prefix)]
    stats_by_user = (
        data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param]]
        .reset_index(drop=True)
    )
    e_theta_star = stats_by_user[param].quantile(0.5, interpolation='lower')
    e_x_theta_star = stats_by_user[metric].median()
    lo, hi = stats_by_user[param].quantile([0.025, 0.975])
    in_interval = data.loc[data[param].between(lo, hi)]
    std_by_delta = in_interval.groupby(param)[metric].std()
    mean_metric_by_param = in_interval.groupby(param)[metric].mean()
    return {
        "beta_ping": beta_ping,
        "ha": ha,
        "med_optimal_param": e_theta_star,
        "med_best_metric": round(e_x_theta_star, 4),
        "95pct_param_range": f"[{lo:.1f}, {hi:.1f}]",
        "std_at_optimal_param": std_by_delta.loc[e_theta_star],
        "max_std_in_interval": std_by_delta.max(),
        "min_std_in_interval": std_by_delta.min(),
        "max_mean_performance": mean_metric_by_param.max(),
        "min_interval_performance": mean_metric_by_param.min()
    }

In [40]:
config_files = ['config_ha13_beta5.json', 'config_ha13_beta6.json',
                'config_ha13_beta7.json', 'config_ha13_beta8.json',
                'config_ha15_beta5.json', 'config_ha15_beta6.json',
                'config_ha15_beta7.json', 'config_ha15_beta8.json',
                'config_ha17_beta5.json', 'config_ha17_beta6.json',
                'config_ha17_beta7.json', 'config_ha17_beta8.json']

POI_LOCATIONS = ['w-x17-y10', 'r-x19-y11']

dist_thresh_values = np.linspace(1, 60, 5)
delta_roam_values  = np.linspace(5, 80, 10)

algos = [*[("seqscan",{"func": seqscan_labels_per_user,
                      "params": {"time_thresh": 45, "dist_thresh": dr, "min_pts": 2,
                                 "back_merge": False, "n_jobs": 2},
                                 "visit_attr": "majority"}) for dr in dist_thresh_values],
        *[("seqscan", {"func": seqscan_labels_per_user,
                       "params": {"time_thresh": 45, "dist_thresh": dr, "min_pts": 3,
                                  "back_merge": False, "n_jobs": 2},
                                  "visit_attr": "majority"}) for dr in dist_thresh_values],
        *[("lachesis", {"func": lachesis_labels_per_user,
                        "params": {"dt_max": 45, "delta_roam": 1.7 * dr, "n_jobs": 2},
                        "visit_attr": "majority"}) for dr in delta_roam_values]]

summarize_stops_with_loc = partial(
    utils.summarize_stop,
    x='x', y='y', timestamp='timestamp',
    keep_col_names=True,
    passthrough_cols=['location', 'user_id', 'cluster'],
    complete_output=False
)

In [42]:
all_results = []

for config_file in config_files:
    key = config_file.replace('config_', '').replace('.json', '')

    if os.path.exists(f"results_{key}.parquet"):
        print(f"results_{key}.parquet already exists. Skipping save.")
        all_results.append(pd.read_parquet(f"results_{key}.parquet"))
        continue

    with open(config_file, 'r', encoding='utf-8') as f:
        config = json.load(f)

    beta_ping = config['agent_params']['beta_ping']
    ha = config['agent_params']['ha'] * 15

    poi_table  = gpd.read_parquet(config["buildings_file"]).rename(columns={"id": "location"})
    sparse_df  = loader.from_file(config["output_files"]["sparse_path"], format="parquet")
    sparse_df.drop(columns=['datetime'], inplace=True)
    diaries_df = loader.from_file(config["output_files"]["diaries_path"], format="parquet").rename(
        columns={"identifier": "user_id"})

    poi_subset = poi_table.loc[poi_table.location.isin(POI_LOCATIONS)]

    sparse_df['precomp_locations'] = visits.poi_map(
        sparse_df, poi_table=poi_subset, data_crs='EPSG:3857',
        max_distance=20, location_id='location', x='x', y='y'
    )

    results_list = []
    for algo_name, algo_config in tqdm(algos, desc=key):
        labels = algo_config["func"](sparse_df, **algo_config["params"],
                                     x="x", y="y", timestamp='timestamp',
                                     user_id='user_id')
        sparse_df['cluster']  = labels
        sparse_df['location'] = sparse_df['precomp_locations']

        sparse_df['location'] = visits.point_in_polygon(
            sparse_df, poi_table=poi_subset, data_crs='EPSG:3857',
            max_distance=20, location_id='location',
            method=algo_config["visit_attr"], x='x', y='y',
            recompute_location=False
        )

        stops_all = (
            sparse_df[sparse_df.cluster != -1]
            .groupby(['user_id', 'cluster'], as_index=False)
            .apply(summarize_stops_with_loc, include_groups=False).reset_index()
        )

        for user in diaries_df.user_id.unique():
            stops = stops_all[stops_all.user_id == user]
            truth = diaries_df.query("user_id==@user")

            metrics = compute_stop_detection_metrics(
                stops=stops, truth=truth, user_id=user, algorithm=algo_name,
                traj_cols={'location_id': 'location'}, timestamp='timestamp'
            )
            metrics['key']        = key
            metrics['beta_ping']  = beta_ping
            metrics['ha']         = ha
            metrics['dist_thresh'] = algo_config["params"].get("dist_thresh", np.nan)
            metrics['delta_roam']  = algo_config["params"].get("delta_roam", np.nan)
            metrics['min_pts'] = algo_config["params"].get("min_pts", np.nan)
            results_list.append(metrics)

        sparse_df.drop(columns=['cluster', 'location'], inplace=True)

    config_df = pd.DataFrame(results_list)
    config_df.to_parquet(f"results_{key}.parquet", index=False)
    all_results.append(config_df)
    print(f"  {len(config_df)} rows saved to results_{key}.parquet")

results_df = pd.concat(all_results, ignore_index=True)
results_df.to_parquet("results_all_configs.parquet", index=False)
print(f"\nTotal rows: {len(results_df)}")

ha13_beta5: 100%|██████████| 20/20 [00:05<00:00,  3.88it/s]


  200 rows saved to results_ha13_beta5.parquet


ha13_beta6: 100%|██████████| 20/20 [00:02<00:00,  8.75it/s]


  200 rows saved to results_ha13_beta6.parquet


ha13_beta7: 100%|██████████| 20/20 [00:02<00:00,  8.94it/s]


...

  200 rows saved to results_ha17_beta6.parquet


ha17_beta7: 100%|██████████| 20/20 [00:02<00:00,  8.54it/s]


  200 rows saved to results_ha17_beta7.parquet


ha17_beta8: 100%|██████████| 20/20 [00:02<00:00,  9.01it/s]

  200 rows saved to results_ha17_beta8.parquet

Total rows: 2400


In [30]:

# def stats_summary(data, metric="recall", param='delta_roam', param2=None):
#     data = data.dropna(subset=[param])
    
#     # bulk performance
#     if param2:
#         stats_by_user = (data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param, param2, "ha", "beta_ping"]].reset_index(drop=True))
#         param2_star = stats_by_user[param2].value_counts().idxmax()
#         lo, hi = stats_by_user[stats_by_user[param2] == param2_star][param].quantile([0.025, 0.975])
#         theta_star = stats_by_user[stats_by_user[param2] == param2_star][param].quantile(0.5, interpolation='lower')
#         # mean_acc_theta_star = stats_by_user[stats_by_user[param2] == param2_star][param].mean()
#         # std_acc_theta_star = stats_by_user[stats_by_user[param2] == param2_star][param].std()
#         in_interval = data.loc[data[param].between(lo, hi) & (data[param2] == param2_star)]
#     else:
#         stats_by_user = (data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param, "ha", "beta_ping"]].reset_index(drop=True))
#         lo, hi = stats_by_user[param].quantile([0.025, 0.975])
#         theta_star = stats_by_user[param].quantile(0.5, interpolation='lower')
#         # mean_acc_theta_star = stats_by_user[param].mean()
#         # std_acc_theta_star = stats_by_user[param].std()
#         in_interval = data.loc[data[param].between(lo, hi)]
    
#     std_by_delta = in_interval.groupby(param)[metric].std()
#     mean_metric_by_param = in_interval.groupby(param)[metric].mean()

#     return pd.DataFrame({ "ha": stats_by_user["ha"].iloc[0],
#                          "beta_ping": stats_by_user["beta_ping"].iloc[0],
#                          "mean_acc_theta_star": theta_star.mean(),
#                          "mean_acc_theta_star_i": round(stats_by_user[metric].mean(), 4), # oracle performance
#                          "std_acc_theta_star": theta_star.std(),
#                          "95pct_param_range": f"[{lo:.1f}, {hi:.1f}]",
#                          "std_at_optimal_param": std_by_delta.loc[theta_star],
#                          "max_std_in_interval": std_by_delta.max(),
#                          "min_std_in_interval": std_by_delta.min(),
#                          "max_mean_performance": mean_metric_by_param.max(),
#                          "min_interval_performance": mean_metric_by_param.min()}, index=[param])



def stats_summary(data, metric="recall", param='delta_roam', param2=None):
    data = data.dropna(subset=[param])
    
    if param2:
        stats_by_user = (data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param, param2, "ha", "beta_ping"]].reset_index(drop=True))
        param2_star = stats_by_user[param2].value_counts().idxmax()
        theta_star = stats_by_user[stats_by_user[param2] == param2_star][param].quantile(0.5, interpolation='lower')
        mean_acc_theta_star = data.loc[data[param]==theta_star][metric].mean()
        std_acc_theta_star = data.loc[data[param]==theta_star][metric].std()
        
    else:
        stats_by_user = (data.loc[data.groupby("user_id")[metric].idxmax(), ["user_id", metric, param, "ha", "beta_ping"]].reset_index(drop=True))
        theta_star = stats_by_user[param].quantile(0.5, interpolation='lower')
        mean_acc_theta_star = data.loc[data[param]==theta_star][metric].mean()
        std_acc_theta_star = data.loc[data[param]==theta_star][metric].std()

    return pd.DataFrame({"algorithm": data['algorithm'].iloc[0],
                         "ha": stats_by_user["ha"].iloc[0],
                         "beta_ping": stats_by_user["beta_ping"].iloc[0],
                         "mean_acc_theta_star": mean_acc_theta_star,
                         "mean_acc_theta_star_i": round(stats_by_user[metric].mean(), 4), # oracle performance
                         "std_acc_theta_star": std_acc_theta_star,
                         "theta_star": theta_star},
                         index=[param])

In [35]:
results_df

,precision,recall,f1,missed_fraction,merged_fraction,split_fraction,user_id,algorithm,key,beta_ping,ha,dist_thresh,delta_roam,min_pts
0,0.000000,0.000000,0.000000,1.0,0.0,0.0,festive_thirsty_tesla,seqscan,ha13_beta5,5,13.0,1.0,NaN,2.0
1,0.000000,0.000000,0.000000,1.0,0.0,0.0,flamboyant_hungry_feynman,seqscan,ha13_beta5,5,13.0,1.0,NaN,2.0
2,0.000000,0.000000,0.000000,1.0,0.0,0.0,flamboyant_priceless_pike,seqscan,ha13_beta5,5,13.0,1.0,NaN,2.0
3,0.000000,0.000000,0.000000,1.0,0.0,0.0,flamboyant_xenodochial_chatterjee,seqscan,ha13_beta5,5,13.0,1.0,NaN,2.0
4,1.000000,0.064246,0.120735,NaN,NaN,NaN,focused_lucid_meitner,seqscan,ha13_beta5,5,13.0,1.0,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2395,0.489489,0.455307,0.471780,NaN,NaN,NaN,focused_sleepy_benz,lachesis,ha17_beta8,8,17.0,NaN,136.0,NaN
2396,0.498559,0.483240,0.490780,NaN,NaN,NaN,friendly_happy_lichterman,lachesis,ha17_beta8,8,17.0,NaN,136.0,NaN
2397,0.501416,0.494413,0.497890,NaN,NaN,NaN,friendly_pedantic_babbage,lachesis,ha17_beta8,8,17.0,NaN,136.0,NaN
2398,0.498567,0.486034,0.492221,NaN,NaN,NaN,friendly_vigilant_leakey,lachesis,ha17_beta8,8,17.0,NaN,136.0,NaN


In [36]:
results1 = results_df.groupby("key").apply(stats_summary, metric="recall", param='delta_roam').reset_index().drop(columns=['key', 'level_1'])
results2 = results_df.groupby("key").apply(stats_summary, metric="recall", param='dist_thresh', param2='min_pts').reset_index().drop(columns=['key', 'level_1'])

/var/folders/nt/0tc5pmb17xd73rr4g1xb0g_00000gn/T/ipykernel_6619/2333220795.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results1 = results_df.groupby("key").apply(stats_summary, metric="recall", param='delta_roam').reset_index().drop(columns=['key', 'level_1'])
/var/folders/nt/0tc5pmb17xd73rr4g1xb0g_00000gn/T/ipykernel_6619/2333220795.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results2 = results_df.gr

In [37]:
results1

,algorithm,ha,beta_ping,mean_acc_theta_star,mean_acc_theta_star_i,std_acc_theta_star,theta_star
0,lachesis,13.0,5,0.920112,0.9310,0.045580,51.000000
1,lachesis,13.0,6,0.906425,0.9176,0.045922,51.000000
2,lachesis,13.0,7,0.873464,0.8835,0.113243,65.166667
3,lachesis,13.0,8,0.843575,0.8469,0.122502,65.166667
4,lachesis,15.0,5,0.929330,0.9408,0.043594,65.166667
5,lachesis,15.0,6,0.913966,0.9173,0.037425,65.166667
6,lachesis,15.0,7,0.873464,0.8796,0.113243,65.166667
7,lachesis,15.0,8,0.840782,0.8480,0.123671,65.166667
8,lachesis,17.0,5,0.927933,0.9327,0.041532,65.166667
9,lachesis,17.0,6,0.911732,0.9182,0.040117,65.166667


In [38]:
results2

,algorithm,ha,beta_ping,mean_acc_theta_star,mean_acc_theta_star_i,std_acc_theta_star,theta_star
0,seqscan,13.0,5,0.896369,0.9168,0.053223,15.75
1,seqscan,13.0,6,0.811592,0.8581,0.130557,15.75
2,seqscan,13.0,7,0.742318,0.8737,0.194448,30.50
3,seqscan,13.0,8,0.722207,0.8285,0.215713,30.50
4,seqscan,15.0,5,0.883520,0.9081,0.052282,15.75
5,seqscan,15.0,6,0.778212,0.8327,0.141770,15.75
6,seqscan,15.0,7,0.732821,0.8721,0.199286,30.50
7,seqscan,15.0,8,0.764385,0.8159,0.209728,30.50
8,seqscan,17.0,5,0.804609,0.8492,0.130107,15.75
9,seqscan,17.0,6,0.629190,0.8109,0.211680,30.50


In [39]:
pd.concat([results1, results2], ignore_index=True).sort_values(['beta_ping', 'ha'])

,algorithm,ha,beta_ping,mean_acc_theta_star,mean_acc_theta_star_i,std_acc_theta_star,theta_star
0,lachesis,13.0,5,0.920112,0.9310,0.045580,51.000000
12,seqscan,13.0,5,0.896369,0.9168,0.053223,15.750000
4,lachesis,15.0,5,0.929330,0.9408,0.043594,65.166667
16,seqscan,15.0,5,0.883520,0.9081,0.052282,15.750000
8,lachesis,17.0,5,0.927933,0.9327,0.041532,65.166667
20,seqscan,17.0,5,0.804609,0.8492,0.130107,15.750000
1,lachesis,13.0,6,0.906425,0.9176,0.045922,51.000000
13,seqscan,13.0,6,0.811592,0.8581,0.130557,15.750000
5,lachesis,15.0,6,0.913966,0.9173,0.037425,65.166667
17,seqscan,15.0,6,0.778212,0.8327,0.141770,15.750000
